In [12]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [13]:
exp_name = "estimate1122"
node_name = "route_10"
es_scale = 625
time_ratio = 15 / 60

In [14]:
es_info = {"transform_capacity": 8883000,
            "invertband": 0,
            "soc_redundant_ratio": 0,
            "usable_depth": 0.90,
            "charge_loss": 0.92,
            "discharge_loss": 0.95,
            "es_charge_max": es_scale,
            "es_charge_min": -es_scale,
            "es_capacity_max": 1305,
            "es_capacity_min": 0}

In [15]:
def split_load_by_month(df):
    """
    将以时间戳为索引、包含'value'列的DataFrame按月拆分，返回一个字典。
    
    参数:
        df (pd.DataFrame): 索引为时间对象（datetime-like），包含'value'列。
    
    返回:
        dict: 键为'YYYY-MM'格式的字符串，值为对应月份的'value' Series。
    """
    # 确保索引是 datetime 类型
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index)
    
    # 按月分组
    monthly_groups = df.groupby(df.index.to_period('M'))
    
    # 构建字典：key 为 'YYYY-MM' 字符串，value 为该月的 'value' Series
    result = {
        str(period): group
        for period, group in monthly_groups
    }
    
    return result

def get_days_in_month(date_str):
    try:
        year, month = map(int, date_str.split('-'))
        # calendar.monthrange(year, month) 返回 (weekday_of_first_day, number_of_days)
        _, days = calendar.monthrange(year, month)
        return days
    except ValueError as e:
        print(f"输入格式错误或无效日期: {e}")
        return None

In [16]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/es_scale_experiment/schedule_result_scale_{es_scale}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [17]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)

In [18]:
month_load_dict = split_load_by_month(es_charge_df)

In [35]:
month_discharge_dict = dict()
for k,v in month_load_dict.items():
    year_num = int(k.split('-')[0])
    month_num = int(k.split('-')[1])
    days_in_month = calendar.monthrange(year_num, month_num)[1]
    discharge_amont = v[v['value'] > 0]['value'].sum() * time_ratio
    achieve_ratio = discharge_amont / (1115.775 * 2 * days_in_month)
    print(k, round(achieve_ratio * 100, 2), "%")

2024-07 68.31 %
2024-08 66.51 %
2024-09 77.93 %
2024-10 98.53 %
2024-11 99.57 %
2024-12 71.84 %
2025-01 80.04 %
2025-02 94.72 %
2025-03 95.98 %
2025-04 97.23 %
2025-05 96.17 %
2025-06 98.13 %


In [23]:
year_num

2025

In [27]:
v

,value
2025-06-01 00:00:00,-493.461
2025-06-01 00:15:00,0.000
2025-06-01 00:30:00,0.000
2025-06-01 00:45:00,-500.642
2025-06-01 01:00:00,0.000
...,...
2025-06-30 22:45:00,0.000
2025-06-30 23:00:00,0.000
2025-06-30 23:15:00,0.000
2025-06-30 23:30:00,0.000


In [29]:
4463.1 / 4

1115.775

In [30]:
1305 * 0.9 * 0.95

1115.7749999999999